# Embed the Topic Definition

| Item | Description |
|---|---|
| Goal | Turn the codebook's label and definition of Topic 68, consumer data privacy legislation, into one unit-length embedding: topic vector c, Option 1. |
| Input | `codebook_v1.parquet`, the frozen codebook of T2.1.4, in the T2.1 Processed folder on the scratch root. |
| Output | `output/T3.5.1/topic_definition.csv` (the text and its token count) and `output/T3.5.1/topic_definition_embedding.npy` (1 × 1024, float32). |
| Unit | One topic. |
| Model | `BAAI/bge-large-en-v1.5` from the project's local snapshot: the CLS token of the last layer, L2-normalized, at most 512 input tokens. Run on the GPU in float32. |
| Variables | • `topic_text`: the codebook `label`, a colon and a space, then the codebook `definition`.<br>• `topic_definition_embedding`: $c_{\text{def}} = e(\text{topic\_text}) / \lVert e(\text{topic\_text}) \rVert \in \mathbb{R}^{1024}$, where $e$ is the CLS embedding. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for how this vector is used downstream.


## Step 1. Import packages

Read the codebook, load the encoder, and save the vector.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from IPython.display import display


## Step 2. Define file locations

Read the codebook and the model snapshot in place; save outputs in `output/T3.5.1/` under the TEMP folder.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.5.1"  # the identifier names this notebook's output folder
CODEBOOK_FILE = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis") / "T2 - Topic Extraction and Stance Analysis/T2.1 - Track 1 - Topic Extraction based on Hearing Summaries" / "Processed" / "codebook_v1.parquet"  # the frozen codebook of T2.1.4
MODEL_FOLDER = Path(
    "/scratch2/acroh/projects/llm_method/hf_cache/hub/models--BAAI--bge-large-en-v1.5"
    "/snapshots/d4aa6901d3a41ba39fb536a557fa166f842b0e09"
)
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

TOPIC_ID = 68  # consumer data privacy legislation
MAX_TOKENS = 512  # the encoder's input limit
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.1


## Step 3. Load the topic's codebook row

Read the frozen codebook and keep the one row for Topic 68. Show its label and definition.


In [3]:
codebook = pd.read_parquet(CODEBOOK_FILE)
assert codebook["topic_id"].is_unique
topic = codebook.loc[codebook["topic_id"].eq(TOPIC_ID)]
assert len(topic) == 1
topic = topic.iloc[0]
print(f"Codebook rows: {len(codebook):,}")
print(f"Label: {topic['label']}")
print(f"Definition: {topic['definition']}")


Codebook rows: 111
Label: consumer data privacy legislation
Definition: This topic encompasses federal and state laws, proposed bills, and regulatory frameworks specifically addressing the collection, use, and protection of consumer personal information.


## Step 4. Build the topic text

The text to embed is the label, a colon, and the definition, so the vector carries both the topic's name and what it covers.


In [4]:
topic_text = f"{topic['label']}: {topic['definition']}"
print(topic_text)


consumer data privacy legislation: This topic encompasses federal and state laws, proposed bills, and regulatory frameworks specifically addressing the collection, use, and protection of consumer personal information.


## Step 5. Load the encoder

Load the tokenizer and model from the local snapshot only, in evaluation mode on the GPU. Check that the model's input limit is the 512 tokens assumed throughout.


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_FOLDER, local_files_only=True)
assert torch.cuda.is_available(), "run this notebook on a GPU node"
device = torch.device("cuda")
model = AutoModel.from_pretrained(MODEL_FOLDER, local_files_only=True).to(device).eval()
assert model.config.max_position_embeddings == MAX_TOKENS
print(f"Encoder: {MODEL_FOLDER.parent.parent.name}; hidden size {model.config.hidden_size}; input limit {MAX_TOKENS} tokens")
print(f"Device: {torch.cuda.get_device_name(device)}; dtype {next(model.parameters()).dtype}")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Encoder: models--BAAI--bge-large-en-v1.5; hidden size 1024; input limit 512 tokens
Device: NVIDIA A40; dtype torch.float32


## Step 6. Count the tokens

The text must fit the encoder's input limit; otherwise it would be cut off silently.


In [6]:
n_tokens = len(tokenizer(topic_text)["input_ids"])
assert n_tokens <= MAX_TOKENS
print(f"Tokens including the special start and end tokens: {n_tokens}")


Tokens including the special start and end tokens: 36


## Step 7. Embed the text

Run the encoder once, take the CLS token of the last layer, and scale it to unit length so that cosine similarity downstream is a plain dot product.


In [7]:
with torch.no_grad():
    encoded = tokenizer(topic_text, truncation=True, max_length=MAX_TOKENS, return_tensors="pt").to(device)
    last_layer = model(**encoded).last_hidden_state
    cls_vector = last_layer[:, 0]
    topic_definition_embedding = torch.nn.functional.normalize(cls_vector, dim=-1).cpu().numpy().astype(np.float32)
assert topic_definition_embedding.shape == (1, model.config.hidden_size)
assert abs(np.linalg.norm(topic_definition_embedding) - 1) < 1e-5
print(f"Shape: {topic_definition_embedding.shape}; norm: {np.linalg.norm(topic_definition_embedding):.6f}")
print(f"First five values: {np.round(topic_definition_embedding[0, :5], 4)}")


Shape: (1, 1024); norm: 1.000000
First five values: [-0.0186 -0.0354  0.019   0.0116  0.0071]


## Step 8. Save the text and the vector

Save the text with its token count beside the vector so a reader can see exactly what was embedded.


In [8]:
text_file = OUTPUT_FOLDER / "topic_definition.csv"
embedding_file = OUTPUT_FOLDER / "topic_definition_embedding.npy"
pd.DataFrame([{
    "topic_id": TOPIC_ID,
    "label": topic["label"],
    "definition": topic["definition"],
    "topic_text": topic_text,
    "n_tokens": n_tokens,
}]).to_csv(text_file, index=False)
np.save(embedding_file, topic_definition_embedding)
print(f"Saved: {text_file}")
print(f"Saved: {embedding_file}")
display(pd.read_csv(text_file))


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.1/topic_definition.csv
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.1/topic_definition_embedding.npy


,topic_id,label,definition,topic_text,n_tokens
0,68,consumer data privacy legislation,"This topic encompasses federal and state laws,...",consumer data privacy legislation: This topic ...,36
